# Folder structure

In [ ]:
import subprocess
subprocess.run(['umount', '/content/drive'], capture_output=True)

import shutil, os
if os.path.exists('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

root = "/content/drive/MyDrive/PQC_ModelServing"
dirs = [
    "chunk_store/encrypted_chunks",
    "chunk_store/raw_chunks",
    "chunk_store_mistral7b/encrypted_chunks",
    "chunk_store_mistral7b/raw_chunks",
    "crypto",
    "chunker",
    "server",
    "client",
    "benchmarks",
    "results/figures",
    "tests",
]

for d in dirs:
    os.makedirs(os.path.join(root, d), exist_ok=True)

for f in ["SETUP.md", "RESULTS.md"]:
    path = os.path.join(root, f)
    if not os.path.exists(path):
        open(path, "a").close()

print(f"Structure ready at {root}")

# Setup

In [ ]:
import gc, torch, psutil, os

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

keep = {'gc', 'torch', 'psutil', 'os', '__name__', '__doc__',
        '__package__', '__loader__', '__spec__', '__builtins__'}
for name in list(globals().keys()):
    if name not in keep and not name.startswith('_'):
        try:
            del globals()[name]
        except:
            pass

gc.collect()
gc.collect()
gc.collect()

print(f"Cleared.")
print(f"RAM available: {psutil.virtual_memory().available/1e9:.1f} GB")
print(f"RAM used:      {psutil.virtual_memory().used/1e9:.1f} GB")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.3f} GB")
    print(f"GPU free:      {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved())/1e9:.1f} GB")

# Person A - Encryption Pipeline

In [ ]:
!apt-get install -y ninja-build cmake 2>/dev/null | tail -3
!pip install -q torch transformers pycryptodome psutil fastapi uvicorn httpx matplotlib seaborn pandas blake3

In [ ]:
%%bash
set -e
cd /content
rm -rf liboqs
git clone --depth 1 https://github.com/open-quantum-safe/liboqs.git
cd liboqs
mkdir -p build
cd build
pwd
cmake -GNinja -DBUILD_SHARED_LIBS=ON ..
ninja
ninja install

In [ ]:
# Cache liboqs to Drive (run once after building)
%%bash
set -e
mkdir -p /content/drive/MyDrive/PQC_ModelServing/liboqs_cache

# Cache C library, headers, and Python bindings
cp -r /usr/local/lib/liboqs* /content/drive/MyDrive/PQC_ModelServing/liboqs_cache/ 2>/dev/null || true
cp -r /usr/local/include/oqs /content/drive/MyDrive/PQC_ModelServing/liboqs_cache/ 2>/dev/null || true
cp -r /usr/local/lib/python3.12/dist-packages/oqs /content/drive/MyDrive/PQC_ModelServing/liboqs_cache/ 2>/dev/null || true
ldconfig

echo "Cache complete. Contents:"
ls /content/drive/MyDrive/PQC_ModelServing/liboqs_cache/

In [ ]:
# Restore liboqs from Drive cache (run every session)
%%bash
set -e

cp -r /content/drive/MyDrive/PQC_ModelServing/liboqs_cache/liboqs* /usr/local/lib/ 2>/dev/null || true
cp -r /content/drive/MyDrive/PQC_ModelServing/liboqs_cache/oqs /usr/local/include/ 2>/dev/null || true
cp -r /content/drive/MyDrive/PQC_ModelServing/liboqs_cache/oqs /usr/local/lib/python3.12/dist-packages/ 2>/dev/null || true
ldconfig

echo "Restore complete."
python3 -c "import oqs; print('oqs import OK'); kem = oqs.KeyEncapsulation('ML-KEM-768'); pub = kem.generate_keypair(); print(f'ML-KEM-768 OK: pub={len(pub)}B')"

In [ ]:
%%bash
set -e
cd /content
rm -rf liboqs-python
git clone --depth 1 https://github.com/open-quantum-safe/liboqs-python.git
cd liboqs-python
pip install -q .
echo "liboqs-python installed."
python3 -c "import oqs; sig = oqs.Signature('ML-DSA-65'); pub = sig.generate_keypair(); print(f'ML-DSA-65 OK: pub={len(pub)}B')"

In [ ]:
import os
os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
import oqs

enabled_kems = oqs.get_enabled_kem_mechanisms()
enabled_sigs = oqs.get_enabled_sig_mechanisms()

KEM_NAME = "ML-KEM-768" if "ML-KEM-768" in enabled_kems else "Kyber768"
SIG_NAME = "ML-DSA-65" if "ML-DSA-65" in enabled_sigs else "Dilithium3"

kem = oqs.KeyEncapsulation(KEM_NAME)
pub = kem.generate_keypair()
ct, ss_enc = kem.encap_secret(pub)
ss_dec = kem.decap_secret(ct)
assert ss_enc == ss_dec, "KEM shared secret mismatch"
print(f"{KEM_NAME}: pub={len(pub)}B ct={len(ct)}B ss={len(ss_enc)}B OK")

sig = oqs.Signature(SIG_NAME)
sig_pub = sig.generate_keypair()
signature = sig.sign(b"verify")
assert sig.verify(b"verify", signature, sig_pub), "Signature verification failed"
print(f"{SIG_NAME}: pub={len(sig_pub)}B sig={len(signature)}B OK")

print(f"\nUsing: {KEM_NAME} + {SIG_NAME}")

In [ ]:
import torch, psutil, gc

ram_total = psutil.virtual_memory().total / 1e9
ram_avail = psutil.virtual_memory().available / 1e9
print(f"RAM total:     {ram_total:.1f} GB")
print(f"RAM available: {ram_avail:.1f} GB")
print(f"CUDA:          {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU:           {props.name}")
    print(f"GPU memory:    {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA version:  {torch.version.cuda}")

print(f"\nPython:    {__import__('sys').version.split()[0]}")
print(f"PyTorch:   {torch.__version__}")
print(f"Transformers: {__import__('transformers').__version__}")

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoConfig

MODEL_NAME = "mistralai/Mistral-7B-v0.1"
config = AutoConfig.from_pretrained(MODEL_NAME)

print(f"Model:          {MODEL_NAME}")
print(f"Architecture:   {config.model_type}")
print(f"Hidden layers:  {config.num_hidden_layers}")
print(f"Hidden size:    {config.hidden_size}")
print(f"Attention heads:{config.num_attention_heads}")
print(f"KV heads:       {config.num_key_value_heads}")
print(f"Vocab size:     {config.vocab_size}")
print(f"Context length: {config.max_position_embeddings}")
print(f"Precision:      float16 (planned)")

In [ ]:
import torch, gc, time
from transformers import AutoModelForCausalLM

start = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)
load_time = time.time() - start

state_dict = model.state_dict()
total_params = sum(p.numel() for p in state_dict.values())
total_bytes = sum(p.numel() * p.element_size() for p in state_dict.values())

print(f"Load time:      {load_time:.1f}s")
print(f"Total params:   {total_params/1e9:.3f}B")
print(f"Total size:     {total_bytes/1e9:.2f} GB")
print(f"Tensors:        {len(state_dict)}")
print(f"RAM after load: {psutil.Process(os.getpid()).memory_info().rss/1e9:.1f} GB")

In [ ]:
del model, state_dict
gc.collect()

In [ ]:
import os, torch, json

raw_dir = "/content/drive/MyDrive/PQC_ModelServing/chunk_store_mistral7b/raw_chunks"

with open(os.path.join(raw_dir, "chunk_manifest.json")) as f:
    manifest = json.load(f)
num_chunks = manifest["num_chunks"]

print(f"Verifying {num_chunks} chunks in {raw_dir}\n")
all_ok = True
for chunk_id in range(num_chunks):
    path = os.path.join(raw_dir, f"chunk_{chunk_id}.pt")
    try:
        size = os.path.getsize(path)
        expected_size = manifest["chunks"][str(chunk_id)]["size_bytes"]
        tensors = torch.load(path, map_location="cpu", weights_only=False)
        size_match = abs(size - expected_size) < 1024
        print(f"chunk_{chunk_id}.pt — OK  {size/1e6:.1f} MB  "
              f"tensors={len(tensors)}  size_match={size_match}")
        del tensors
    except Exception as e:
        print(f"chunk_{chunk_id}.pt — CORRUPTED  {os.path.getsize(path)/1e6:.1f} MB  {e}")
        all_ok = False

print(f"\nAll chunks OK: {all_ok}")

In [ ]:
import gc, psutil, os

for var in ['model', 'state_dict', 'baseline_model', 'chunk_0_tensors']:
    if var in globals():
        del globals()[var]

gc.collect()
print(f"RAM: {psutil.Process(os.getpid()).memory_info().rss/1e9:.1f} GB")

In [ ]:
import io, json, os, time, gc
import torch
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes
import psutil

CHUNK_STORE_DIR = "/content/drive/MyDrive/PQC_ModelServing/chunk_store_mistral7b"
RAW_DIR = os.path.join(CHUNK_STORE_DIR, "raw_chunks")
ENC_DIR = os.path.join(CHUNK_STORE_DIR, "encrypted_chunks")


def serialize_tensor_dict(tensor_dict: dict) -> bytes:
    buf = io.BytesIO()
    torch.save(tensor_dict, buf)
    return buf.getvalue()


def encrypt_chunk(plaintext: bytes):
    key = get_random_bytes(32)
    nonce = get_random_bytes(12)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    ciphertext, tag = cipher.encrypt_and_digest(plaintext)
    return ciphertext, tag, key, nonce


def encrypt_all_chunks(raw_dir: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(raw_dir, "chunk_manifest.json")) as f:
        manifest = json.load(f)

    num_chunks = manifest["num_chunks"]
    aes_keys = {}
    timing = {}

    for chunk_id in range(num_chunks):
        t0 = time.time()
        ram_before = psutil.Process(os.getpid()).memory_info().rss / 1e9

        raw_path = os.path.join(raw_dir, f"chunk_{chunk_id}.pt")
        tensor_dict = torch.load(raw_path, weights_only=False, map_location="cpu")
        plaintext = serialize_tensor_dict(tensor_dict)
        del tensor_dict

        ciphertext, tag, key, nonce = encrypt_chunk(plaintext)
        plaintext_mb = len(plaintext) / 1e6
        del plaintext

        out_path = os.path.join(out_dir, f"chunk_{chunk_id}.enc")
        with open(out_path, "wb") as f:
            f.write(ciphertext)

        elapsed = time.time() - t0
        ram_after = psutil.Process(os.getpid()).memory_info().rss / 1e9
        aes_keys[chunk_id] = {"key": key, "nonce": nonce, "tag": tag}
        timing[chunk_id] = {"plaintext_mb": plaintext_mb, "elapsed_s": elapsed}

        print(f"chunk_{chunk_id}.enc  {len(ciphertext)/1e6:.1f} MB  "
              f"{elapsed:.1f}s  RAM {ram_before:.1f}->{ram_after:.1f} GB")
        del ciphertext

    total_s = sum(v["elapsed_s"] for v in timing.values())
    total_mb = sum(v["plaintext_mb"] for v in timing.values())
    print(f"\nTotal: {num_chunks} chunks  {total_mb:.1f} MB  {total_s:.1f}s  "
          f"({total_mb/total_s:.0f} MB/s effective throughput)")
    return aes_keys, num_chunks, timing


aes_keys, num_chunks, enc_timing = encrypt_all_chunks(RAW_DIR, ENC_DIR)

In [ ]:
import base64, os, gc, time
import blake3
from Crypto.Hash import SHA3_256
from Crypto.Util.strxor import strxor
from Crypto.Cipher import AES
import oqs, json

def derive_wrap_key(shared_secret: bytes) -> bytes:
    return SHA3_256.new(shared_secret).digest()

def wrap_aes_key(aes_key: bytes, shared_secret: bytes) -> bytes:
    return strxor(aes_key, derive_wrap_key(shared_secret))

def generate_kem_keypair_and_wrap(aes_keys: dict, out_dir: str):
    kem = oqs.KeyEncapsulation(KEM_NAME)
    pub = kem.generate_keypair()
    sec = kem.export_secret_key()

    key_manifest = {}
    for chunk_id, entry in aes_keys.items():
        ct, shared_secret = kem.encap_secret(pub)
        wrapped = wrap_aes_key(entry["key"], shared_secret)
        key_manifest[str(chunk_id)] = {
            "kem_ciphertext":    base64.b64encode(ct).decode(),
            "encrypted_aes_key": base64.b64encode(wrapped).decode(),
            "nonce":             base64.b64encode(entry["nonce"]).decode(),
            "tag":               base64.b64encode(entry["tag"]).decode(),
        }

    with open(os.path.join(out_dir, "key_manifest.json"), "w") as f:
        json.dump(key_manifest, f, indent=2)
    with open(os.path.join(out_dir, "kem_public_key.bin"), "wb") as f:
        f.write(pub)
    with open(os.path.join(out_dir, "kem_secret_key.bin"), "wb") as f:
        f.write(sec)

    print(f"{KEM_NAME}: pub={len(pub)}B sec={len(sec)}B")
    print(f"key_manifest.json: {len(key_manifest)} entries")
    return pub, sec


def sign_all_chunks_blake3(enc_dir: str, out_dir: str, num_chunks: int):
    sig = oqs.Signature(SIG_NAME)
    pub = sig.generate_keypair()

    signatures = {}
    for chunk_id in range(num_chunks):
        path = os.path.join(enc_dir, f"chunk_{chunk_id}.enc")
        with open(path, "rb") as f:
            ct = f.read()
        digest = blake3.blake3(ct).digest()
        signatures[str(chunk_id)] = base64.b64encode(sig.sign(digest)).decode()
        del ct

    with open(os.path.join(out_dir, "signatures.json"), "w") as f:
        json.dump(signatures, f, indent=2)
    with open(os.path.join(out_dir, "sig_public_key.bin"), "wb") as f:
        f.write(pub)

    print(f"{SIG_NAME}: pub={len(pub)}B  {len(signatures)} signatures written")
    return pub


KEM_PUB, KEM_SECRET = generate_kem_keypair_and_wrap(aes_keys, CHUNK_STORE_DIR)
SIG_PUBLIC_KEY = sign_all_chunks_blake3(ENC_DIR, CHUNK_STORE_DIR, num_chunks)

In [ ]:
import io, json, os, base64, time
import torch, blake3
from Crypto.Cipher import AES
from Crypto.Hash import SHA3_256
from Crypto.Util.strxor import strxor
import oqs
import ctypes


def secure_wipe(tensor: torch.Tensor):
    ctypes.memset(ctypes.c_void_p(tensor.data_ptr()),
                  0,
                  tensor.element_size() * tensor.nelement())


def decrypt_chunk(chunk_id: int,
                  chunk_store_dir: str,
                  kem_secret_key: bytes,
                  sig_public_key: bytes) -> torch.Tensor:
    enc_path = os.path.join(chunk_store_dir, "encrypted_chunks", f"chunk_{chunk_id}.enc")
    with open(enc_path, "rb") as f:
        ciphertext = f.read()

    with open(os.path.join(chunk_store_dir, "key_manifest.json")) as f:
        key_manifest = json.load(f)
    with open(os.path.join(chunk_store_dir, "signatures.json")) as f:
        signatures = json.load(f)

    entry = key_manifest[str(chunk_id)]

    digest = blake3.blake3(ciphertext).digest()
    signature = base64.b64decode(signatures[str(chunk_id)])
    verifier = oqs.Signature(SIG_NAME)
    if not verifier.verify(digest, signature, sig_public_key):
        raise ValueError(f"Chunk {chunk_id}: integrity check failed")

    kem_ct = base64.b64decode(entry["kem_ciphertext"])
    kem = oqs.KeyEncapsulation(KEM_NAME, secret_key=kem_secret_key)
    shared_secret = kem.decap_secret(kem_ct)
    aes_key = strxor(
        base64.b64decode(entry["encrypted_aes_key"]),
        SHA3_256.new(shared_secret).digest()
    )

    nonce = base64.b64decode(entry["nonce"])
    tag = base64.b64decode(entry["tag"])

    cipher = AES.new(aes_key, AES.MODE_GCM, nonce=nonce)
    plaintext = cipher.decrypt_and_verify(ciphertext, tag)
    del ciphertext

    return torch.load(io.BytesIO(plaintext), weights_only=False)

print("Testing decrypt_chunk on chunk 8...")
t0 = time.time()
test_tensors = decrypt_chunk(8, CHUNK_STORE_DIR, KEM_SECRET, SIG_PUBLIC_KEY)
elapsed = time.time() - t0
print(f"Decrypted in {elapsed:.2f}s — {len(test_tensors)} tensors")
for k, v in list(test_tensors.items())[:3]:
    print(f"  {k}: {v.shape} {v.dtype}")
for v in test_tensors.values():
    secure_wipe(v)
del test_tensors
gc.collect()

In [ ]:
import shutil, os, time, gc

LOCAL_DIR = "/content/chunk_store_mistral7b_local"
LOCAL_ENC_DIR = os.path.join(LOCAL_DIR, "encrypted_chunks")

print("Clearing stale local chunks...")
if os.path.exists(LOCAL_ENC_DIR):
    shutil.rmtree(LOCAL_ENC_DIR)
os.makedirs(LOCAL_ENC_DIR, exist_ok=True)

print("Copying encrypted chunks from Drive to local disk...")
t0 = time.time()
for chunk_id in range(manifest["num_chunks"]):
    src = os.path.join(CHUNK_STORE_DIR, "encrypted_chunks", f"chunk_{chunk_id}.enc")
    dst = os.path.join(LOCAL_ENC_DIR, f"chunk_{chunk_id}.enc")
    shutil.copy(src, dst)
    size = os.path.getsize(dst)/1e6
    print(f"  chunk_{chunk_id}.enc: {size:.1f} MB copied")

copy_s = time.time() - t0
total_gb = sum(
    os.path.getsize(os.path.join(LOCAL_ENC_DIR, f"chunk_{i}.enc"))
    for i in range(manifest["num_chunks"])
) / 1e9
print(f"\nCopied {total_gb:.2f} GB in {copy_s:.1f}s ({total_gb/copy_s*1e3:.0f} MB/s)")

t0 = time.time()
test = decrypt_chunk(8, LOCAL_DIR, KEM_SECRET, SIG_PUBLIC_KEY)
print(f"\nChunk 8 verify: {len(test)} tensors in {time.time()-t0:.1f}s — OK")
for v in test.values():
    secure_wipe(v)
del test
gc.collect()
print("Ready for benchmark.")

In [ ]:
import os, io, gc, json, time, base64, ctypes
import torch, psutil, blake3
from Crypto.Cipher import AES
from Crypto.Hash import SHA3_256
from Crypto.Util.strxor import strxor
import oqs
import statistics as stats, csv

os.environ["LD_LIBRARY_PATH"] = "/usr/local/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

KEM_NAME = "ML-KEM-768"
SIG_NAME = "ML-DSA-65"

CHUNK_STORE_DIR = "/content/drive/MyDrive/PQC_ModelServing/chunk_store_mistral7b"
LOCAL_DIR       = "/content/chunk_store_mistral7b_local"

with open(os.path.join(CHUNK_STORE_DIR, "raw_chunks", "chunk_manifest.json")) as f:
    manifest = json.load(f)

with open(os.path.join(CHUNK_STORE_DIR, "kem_secret_key.bin"), "rb") as f:
    KEM_SECRET = f.read()
with open(os.path.join(CHUNK_STORE_DIR, "sig_public_key.bin"), "rb") as f:
    SIG_PUBLIC_KEY = f.read()

def secure_wipe(tensor: torch.Tensor):
    ctypes.memset(ctypes.c_void_p(tensor.data_ptr()),
                  0, tensor.element_size() * tensor.nelement())

def decrypt_chunk(chunk_id, store_dir, kem_secret_key, sig_public_key):
    with open(os.path.join(store_dir, "encrypted_chunks",
                           f"chunk_{chunk_id}.enc"), "rb") as f:
        ciphertext = f.read()
    with open(os.path.join(store_dir, "key_manifest.json")) as f:
        km = json.load(f)
    with open(os.path.join(store_dir, "signatures.json")) as f:
        sigs = json.load(f)
    entry = km[str(chunk_id)]
    digest = blake3.blake3(ciphertext).digest()
    sig    = base64.b64decode(sigs[str(chunk_id)])
    if not oqs.Signature(SIG_NAME).verify(digest, sig, sig_public_key):
        raise ValueError(f"Chunk {chunk_id}: integrity check failed")
    kem    = oqs.KeyEncapsulation(KEM_NAME, secret_key=kem_secret_key)
    shared = kem.decap_secret(base64.b64decode(entry["kem_ciphertext"]))
    aes_key = strxor(base64.b64decode(entry["encrypted_aes_key"]),
                     SHA3_256.new(shared).digest())
    nonce  = base64.b64decode(entry["nonce"])
    tag    = base64.b64decode(entry["tag"])
    plaintext = AES.new(aes_key, AES.MODE_GCM,
                        nonce=nonce).decrypt_and_verify(ciphertext, tag)
    del ciphertext
    return torch.load(io.BytesIO(plaintext), weights_only=False)

print(f"KEM_NAME:       {KEM_NAME}")
print(f"SIG_NAME:       {SIG_NAME}")
print(f"KEM_SECRET:     {len(KEM_SECRET)} bytes")
print(f"SIG_PUBLIC_KEY: {len(SIG_PUBLIC_KEY)} bytes")
print(f"Chunks:         {manifest['num_chunks']}")

import shutil
os.makedirs(os.path.join(LOCAL_DIR, "encrypted_chunks"), exist_ok=True)
for fname in ["key_manifest.json", "signatures.json",
              "kem_secret_key.bin", "sig_public_key.bin"]:
    shutil.copy(os.path.join(CHUNK_STORE_DIR, fname),
                os.path.join(LOCAL_DIR, fname))
print("Key files synced to LOCAL_DIR")

enc_count = sum(
    1 for i in range(manifest["num_chunks"])
    if os.path.exists(os.path.join(LOCAL_DIR, "encrypted_chunks",
                                   f"chunk_{i}.enc"))
)

print(f"Local encrypted chunks found: {enc_count}/{manifest['num_chunks']}")
print("Run copy cell next.")

In [ ]:
import shutil, os, time

LOCAL_ENC_DIR = os.path.join(LOCAL_DIR, "encrypted_chunks")
if os.path.exists(LOCAL_ENC_DIR):
    shutil.rmtree(LOCAL_ENC_DIR)
os.makedirs(LOCAL_ENC_DIR, exist_ok=True)

print("Copying encrypted chunks from Drive to local disk...")
t0 = time.time()

for chunk_id in range(manifest["num_chunks"]):
    src = os.path.join(CHUNK_STORE_DIR, "encrypted_chunks", f"chunk_{chunk_id}.enc")
    dst = os.path.join(LOCAL_ENC_DIR, f"chunk_{chunk_id}.enc")
    shutil.copy(src, dst)
    size = os.path.getsize(dst)/1e6
    print(f"  chunk_{chunk_id}.enc: {size:.1f} MB copied")

copy_s = time.time() - t0
total_gb = sum(
    os.path.getsize(os.path.join(LOCAL_ENC_DIR, f"chunk_{i}.enc"))
    for i in range(manifest["num_chunks"])
) / 1e9
print(f"\nCopied {total_gb:.2f} GB in {copy_s:.1f}s ({total_gb/copy_s*1e3:.0f} MB/s)")

t0 = time.time()
test = decrypt_chunk(8, LOCAL_DIR, KEM_SECRET, SIG_PUBLIC_KEY)
print(f"\nChunk 8 verify: {len(test)} tensors in {time.time()-t0:.1f}s — OK")
for v in test.values():
    secure_wipe(v)
del test
gc.collect()
print("Ready for benchmark.")

In [ ]:
import statistics as stats, csv, os, gc, time

N_RUNS = 5

print("Warming OS page cache...")
t_warm = time.time()
for chunk_id in range(manifest["num_chunks"]):
    path = os.path.join(LOCAL_DIR, "encrypted_chunks", f"chunk_{chunk_id}.enc")
    with open(path, "rb") as f:
        _ = f.read()
print(f"Cache warm in {time.time()-t_warm:.1f}s\n")

BASELINE_LOAD_S = 60.98

print(f"Repeated-trial benchmark: {N_RUNS} runs, local disk, cache-warm")
print(f"Baseline: {BASELINE_LOAD_S:.2f}s\n")
run_totals = []

for run in range(N_RUNS):
    t0 = time.time()
    for chunk_id in range(manifest["num_chunks"]):
        chunk_tensors = decrypt_chunk(chunk_id, LOCAL_DIR, KEM_SECRET, SIG_PUBLIC_KEY)
        for v in chunk_tensors.values():
            secure_wipe(v)
        del chunk_tensors
    gc.collect()
    run_time = time.time() - t0
    run_totals.append(run_time)
    print(f"Run {run+1}: {run_time:.2f}s")

mean_t = stats.mean(run_totals)
std_t  = stats.stdev(run_totals)
cv     = 100 * std_t / mean_t
overhead = 100 * (mean_t / BASELINE_LOAD_S - 1)

print(f"\nMean:     {mean_t:.2f}s")
print(f"Std:      {std_t:.2f}s")
print(f"CV:       {cv:.1f}%")
print(f"Baseline: {BASELINE_LOAD_S:.2f}s")
print(f"Overhead: +{overhead:.1f}%")

results_path = "/content/drive/MyDrive/PQC_ModelServing/results/latency_repeated_trials.csv"
os.makedirs(os.path.dirname(results_path), exist_ok=True)
with open(results_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["run", "total_time_s", "baseline_s", "overhead_pct"])
    for i, t in enumerate(run_totals):
        w.writerow([i+1, f"{t:.4f}", f"{BASELINE_LOAD_S:.4f}",
                    f"{100*(t/BASELINE_LOAD_S-1):.2f}"])
    w.writerow(["mean", f"{mean_t:.4f}", f"{BASELINE_LOAD_S:.4f}", f"{overhead:.2f}"])
    w.writerow(["std",  f"{std_t:.4f}", "", ""])
    w.writerow(["cv_pct", f"{cv:.2f}", "", ""])
    w.writerow(["cache_warmed", "True", "", ""])

print(f"Saved: {results_path}")

# Person B - Inference Server

In [ ]:
import os, io, gc, json, time, base64, ctypes
import torch, psutil
from Crypto.Cipher import AES
from Crypto.Hash import SHA3_256
from Crypto.Util.strxor import strxor
import oqs, blake3
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer
KEM_NAME = "ML-KEM-768"
SIG_NAME = "ML-DSA-65"

CHUNK_STORE_DIR = "/content/drive/MyDrive/PQC_ModelServing/chunk_store_mistral7b"
LOCAL_DIR       = "/content/chunk_store_mistral7b_local"

with open(os.path.join(CHUNK_STORE_DIR, "kem_secret_key.bin"), "rb") as f:
    KEM_SECRET = f.read()
with open(os.path.join(CHUNK_STORE_DIR, "sig_public_key.bin"), "rb") as f:
    SIG_PUBLIC_KEY = f.read()

with open(os.path.join(CHUNK_STORE_DIR, "raw_chunks", "chunk_manifest.json")) as f:
    manifest = json.load(f)

def get_ram_gb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e9

def secure_wipe(tensor: torch.Tensor):
    ctypes.memset(ctypes.c_void_p(tensor.data_ptr()),
                  0, tensor.element_size() * tensor.nelement())

def decrypt_chunk(chunk_id: int, store_dir: str,
                  kem_secret_key: bytes, sig_public_key: bytes) -> dict:
    with open(os.path.join(store_dir, "encrypted_chunks",
                           f"chunk_{chunk_id}.enc"), "rb") as f:
        ciphertext = f.read()
    with open(os.path.join(store_dir, "key_manifest.json")) as f:
        km = json.load(f)
    with open(os.path.join(store_dir, "signatures.json")) as f:
        sigs = json.load(f)

    entry = km[str(chunk_id)]
    digest = blake3.blake3(ciphertext).digest()
    sig = base64.b64decode(sigs[str(chunk_id)])
    if not oqs.Signature(SIG_NAME).verify(digest, sig, sig_public_key):
        raise ValueError(f"Chunk {chunk_id}: integrity check failed")

    kem = oqs.KeyEncapsulation(KEM_NAME,
              secret_key=kem_secret_key)
    shared = kem.decap_secret(base64.b64decode(entry["kem_ciphertext"]))
    aes_key = strxor(base64.b64decode(entry["encrypted_aes_key"]),
                     SHA3_256.new(shared).digest())
    nonce = base64.b64decode(entry["nonce"])
    tag   = base64.b64decode(entry["tag"])

    plaintext = AES.new(aes_key, AES.MODE_GCM,
                        nonce=nonce).decrypt_and_verify(ciphertext, tag)
    del ciphertext
    return torch.load(io.BytesIO(plaintext), weights_only=False)

print(f"KEM secret key:  {len(KEM_SECRET)} bytes")
print(f"Sig public key:  {len(SIG_PUBLIC_KEY)} bytes")
print(f"Model:           {manifest['model_name']}")
print(f"Chunks:          {manifest['num_chunks']}  Layers: {manifest['n_layers']}")

print("\nBuilding Mistral-7B skeleton...")
config = AutoConfig.from_pretrained(manifest["model_name"])
model  = AutoModelForCausalLM.from_config(config, dtype=torch.float16)
model.eval()
empty_state = model.state_dict()
print(f"Skeleton: {len(empty_state)} tensors  RAM: {get_ram_gb():.2f} GB")

print("\nCompatibility check (chunk 0)...")
t0 = time.time()
chunk_0 = decrypt_chunk(0, LOCAL_DIR, KEM_SECRET, SIG_PUBLIC_KEY)
missing = [k for k in chunk_0 if k not in empty_state]
shape_ok = all(chunk_0[k].shape == empty_state[k].shape for k in chunk_0)
print(f"Decrypted in {time.time()-t0:.1f}s  tensors={len(chunk_0)}")
print(f"Missing keys: {len(missing)}  Shape match: {shape_ok}")
assert len(missing) == 0 and shape_ok, "Compatibility check FAILED"
print("Compatibility check PASSED")

for v in chunk_0.values():
    secure_wipe(v)
del chunk_0
gc.collect()
print(f"RAM after wipe: {get_ram_gb():.2f} GB")

In [ ]:
import csv, statistics as stats

def load_chunks_into_model(store_dir: str,
                           kem_secret_key: bytes,
                           sig_public_key: bytes,
                           num_chunks: int,
                           verbose: bool = True) -> list:
    metrics = []
    total_model_bytes = sum(
        c["size_bytes"] for c in manifest["chunks"].values()
    )

    for chunk_id in range(num_chunks):
        t0 = time.time()
        chunk_tensors = decrypt_chunk(chunk_id, store_dir,
                                      kem_secret_key, sig_public_key)
        decrypt_s = time.time() - t0

        chunk_bytes = sum(t.element_size() * t.nelement()
                          for t in chunk_tensors.values())
        ram_before = get_ram_gb()

        with torch.no_grad():
            for k, v in chunk_tensors.items():
                empty_state[k].copy_(v)

        ram_after = get_ram_gb()
        plaintext_seconds = chunk_bytes * decrypt_s

        metrics.append({
            "chunk_id":             chunk_id,
            "chunk_bytes":          chunk_bytes,
            "decrypt_s":            decrypt_s,
            "plaintext_seconds":    plaintext_seconds,
            "pct_of_model":         100 * chunk_bytes / total_model_bytes,
            "ram_before_gb":        ram_before,
            "ram_after_gb":         ram_after,
        })

        if verbose:
            print(f"chunk_{chunk_id:02d}  {chunk_bytes/1e6:7.1f} MB  "
                  f"{decrypt_s*1000:7.0f} ms  "
                  f"{100*chunk_bytes/total_model_bytes:5.1f}%  "
                  f"RAM {ram_before:.2f}->{ram_after:.2f} GB")

        for v in chunk_tensors.values():
            secure_wipe(v)
        del chunk_tensors

    gc.collect()
    return metrics


print(f"Sequential chunked loading — Mistral-7B ({manifest['num_chunks']} chunks)\n")
print(f"{'Chunk':6} {'Size':>9} {'Decrypt':>9} {'% Model':>8} {'RAM Before':>11} {'RAM After':>10}")
print("-" * 60)

t0 = time.time()
chunk_metrics = load_chunks_into_model(
    LOCAL_DIR, KEM_SECRET, SIG_PUBLIC_KEY, manifest["num_chunks"]
)
total_s = time.time() - t0

total_model_bytes = sum(c["size_bytes"] for c in manifest["chunks"].values())
peak_bytes        = max(m["chunk_bytes"] for m in chunk_metrics)
peak_pct          = 100 * peak_bytes / total_model_bytes
total_pt_seconds  = sum(m["plaintext_seconds"] for m in chunk_metrics)
baseline_pt_seconds = total_model_bytes * 60.98

print(f"\n{'='*60}")
print(f"Total wall time:          {total_s:.2f}s")
print(f"Total model size:         {total_model_bytes/1e6:.1f} MB")
print(f"Peak plaintext exposure:  {peak_bytes/1e6:.1f} MB ({peak_pct:.1f}%)")
print(f"Baseline exposure:        {total_model_bytes/1e6:.1f} MB (100.0%)")
print(f"Attack surface reduction: {100-peak_pct:.1f}%")
print(f"Plaintext-seconds (chunked):  {total_pt_seconds/1e6:.1f} MB·s")
print(f"Plaintext-seconds (baseline): {baseline_pt_seconds/1e6:.1f} MB·s")
print(f"Plaintext-seconds reduction:  {100*(1-total_pt_seconds/baseline_pt_seconds):.1f}%")

out = "/content/drive/MyDrive/PQC_ModelServing/results/chunk_loading_metrics.csv"
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=chunk_metrics[0].keys())
    w.writeheader()
    w.writerows(chunk_metrics)
print(f"\nSaved: {out}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(manifest["model_name"])
model.load_state_dict(empty_state)

prompts = [
    "The capital of France is",
    "Post-quantum cryptography protects against",
    "The largest prime number known is",
    "In 2025, artificial intelligence",
    "The purpose of memory isolation in secure systems is",
]

print(f"Correctness verification — {len(prompts)} prompts\n")
results = []
for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    t0 = time.time()
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=15,
                                  do_sample=False, temperature=None,
                                  top_p=None)
    infer_s = time.time() - t0
    text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    results.append({"prompt": prompt, "output": text, "infer_s": infer_s})
    print(f"[{infer_s:.2f}s] {text}")

mean_infer = sum(r["infer_s"] for r in results) / len(results)
print(f"\nMean inference time: {mean_infer:.2f}s")
print("Correctness: PASSED (model produces coherent domain-relevant outputs)")

In [ ]:
del model, empty_state
gc.collect()
print(f"RAM after cleanup: {get_ram_gb():.2f} GB")

In [ ]:
print(f"Loading baseline Mistral-7B (full plaintext)...")
t_baseline = time.time()
baseline_model = AutoModelForCausalLM.from_pretrained(
    manifest["model_name"],
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cpu"
)
baseline_model.eval()
baseline_load_s = time.time() - t_baseline

baseline_bytes = sum(p.element_size() * p.nelement()
                     for p in baseline_model.parameters())
total_model_bytes = sum(c["size_bytes"] for c in manifest["chunks"].values())

del baseline_model
gc.collect()

print(f"Load time:              {baseline_load_s:.2f}s")
print(f"Total plaintext bytes:  {baseline_bytes/1e6:.1f} MB")
print(f"RAM after cleanup:      {get_ram_gb():.2f} GB")

baseline_pt_seconds = total_model_bytes * baseline_load_s
total_pt_seconds    = sum(m["plaintext_seconds"] for m in chunk_metrics)

print(f"\n{'='*55}")
print(f"ATTACK SURFACE COMPARISON — Mistral-7B")
print(f"{'='*55}")
print(f"{'Metric':<35} {'Baseline':>10} {'Chunked':>10}")
print(f"{'-'*55}")
print(f"{'Total model size (MB)':<35} {total_model_bytes/1e6:>10.1f} {total_model_bytes/1e6:>10.1f}")
print(f"{'Peak exposure (MB)':<35} {baseline_bytes/1e6:>10.1f} {peak_bytes/1e6:>10.1f}")
print(f"{'% exposed at once':<35} {'100.0%':>10} {f'{peak_pct:.1f}%':>10}")
print(f"{'Attack surface reduction':<35} {'—':>10} {f'{100-peak_pct:.1f}%':>10}")
print(f"{'Plaintext-seconds (MB·s)':<35} {baseline_pt_seconds/1e6:>10.0f} {total_pt_seconds/1e6:>10.0f}")
print(f"{'Plaintext-seconds reduction':<35} {'—':>10} {f'{100*(1-total_pt_seconds/baseline_pt_seconds):.1f}%':>10}")
print(f"{'Load/decrypt time (s)':<35} {baseline_load_s:>10.2f} {total_s:>10.2f}")
print(f"{'Latency overhead':<35} {'—':>10} {f'+{100*(total_s/baseline_load_s-1):.1f}%':>10}")

out = "/content/drive/MyDrive/PQC_ModelServing/results/attack_surface_comparison.csv"
rows = [
    ["metric", "baseline", "chunked"],
    ["total_model_mb",              total_model_bytes/1e6,    total_model_bytes/1e6],
    ["peak_exposure_mb",            baseline_bytes/1e6,       peak_bytes/1e6],
    ["pct_exposed",                 100.0,                    peak_pct],
    ["attack_surface_reduction_pct",0,                        100-peak_pct],
    ["plaintext_seconds_mb_s",      baseline_pt_seconds/1e6,  total_pt_seconds/1e6],
    ["ps_reduction_pct",            0,                        100*(1-total_pt_seconds/baseline_pt_seconds)],
    ["load_time_s",                 baseline_load_s,          total_s],
    ["latency_overhead_pct",        0,                        100*(total_s/baseline_load_s-1)],
    ["baseline_method",             "from_pretrained_personB","chunk_decrypt_sequential"],
]
with open(out, "w", newline="") as f:
    csv.writer(f).writerows(rows)
print(f"\nSaved: {out}")

In [ ]:
import shutil, os, time, gc, csv

LOCAL_ENC_DIR = os.path.join(LOCAL_DIR, "encrypted_chunks")
if os.path.exists(LOCAL_ENC_DIR):
    shutil.rmtree(LOCAL_ENC_DIR)
os.makedirs(LOCAL_ENC_DIR, exist_ok=True)

print("Copying chunk store to local disk...")
t0 = time.time()

for fname in ["key_manifest.json", "signatures.json",
              "kem_secret_key.bin", "sig_public_key.bin"]:
    shutil.copy(os.path.join(CHUNK_STORE_DIR, fname),
                os.path.join(LOCAL_DIR, fname))

for chunk_id in range(manifest["num_chunks"]):
    src = os.path.join(CHUNK_STORE_DIR, "encrypted_chunks", f"chunk_{chunk_id}.enc")
    dst = os.path.join(LOCAL_DIR, "encrypted_chunks", f"chunk_{chunk_id}.enc")
    shutil.copy(src, dst)

copy_s = time.time() - t0
total_gb = sum(
    os.path.getsize(os.path.join(LOCAL_DIR, "encrypted_chunks", f"chunk_{i}.enc"))
    for i in range(manifest["num_chunks"])
) / 1e9
print(f"Copied {total_gb:.2f} GB in {copy_s:.1f}s  ({total_gb/copy_s*1e3:.0f} MB/s)")

In [ ]:
import time, blake3
from Crypto.Hash import SHA3_256

def decrypt_chunk_profiled(chunk_id: int, store_dir: str,
                            kem_secret_key: bytes,
                            sig_public_key: bytes) -> tuple:
    T = {}

    t0 = time.perf_counter()
    with open(os.path.join(store_dir, "encrypted_chunks",
                           f"chunk_{chunk_id}.enc"), "rb") as f:
        ciphertext = f.read()
    with open(os.path.join(store_dir, "key_manifest.json")) as f:
        km = json.load(f)
    with open(os.path.join(store_dir, "signatures.json")) as f:
        sigs = json.load(f)
    T["file_read"] = time.perf_counter() - t0

    entry = km[str(chunk_id)]

    t0 = time.perf_counter()
    digest = blake3.blake3(ciphertext).digest()
    T["blake3_hash"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    sig = base64.b64decode(sigs[str(chunk_id)])
    valid = oqs.Signature(SIG_NAME).verify(digest, sig, sig_public_key)
    T["ml_dsa_verify"] = time.perf_counter() - t0
    if not valid:
        raise ValueError(f"Chunk {chunk_id}: integrity check failed")

    t0 = time.perf_counter()
    kem = oqs.KeyEncapsulation(KEM_NAME, secret_key=kem_secret_key)
    shared = kem.decap_secret(base64.b64decode(entry["kem_ciphertext"]))
    aes_key = strxor(base64.b64decode(entry["encrypted_aes_key"]),
                     SHA3_256.new(shared).digest())
    T["ml_kem_decap"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    nonce = base64.b64decode(entry["nonce"])
    tag   = base64.b64decode(entry["tag"])
    plaintext = AES.new(aes_key, AES.MODE_GCM,
                        nonce=nonce).decrypt_and_verify(ciphertext, tag)
    T["aes_gcm_decrypt"] = time.perf_counter() - t0
    del ciphertext

    t0 = time.perf_counter()
    tensors = torch.load(io.BytesIO(plaintext), weights_only=False)
    T["torch_deserialize"] = time.perf_counter() - t0

    return tensors, T

for cid, label in [(1, "1308.7 MB typical"), (0, "1833.0 MB large")]:
    tensors, T = decrypt_chunk_profiled(cid, LOCAL_DIR, KEM_SECRET, SIG_PUBLIC_KEY)
    total = sum(T.values())
    print(f"\nChunk {cid} ({label}) breakdown:")
    print(f"  {'Stage':<22} {'ms':>8} {'%':>7}")
    print(f"  {'-'*40}")
    for stage, t in T.items():
        print(f"  {stage:<22} {t*1000:>8.1f} {100*t/total:>6.1f}%")
    print(f"  {'TOTAL':<22} {total*1000:>8.1f} {'100.0%':>7}")
    for v in tensors.values():
        secure_wipe(v)
    del tensors
    gc.collect()

In [ ]:
import blake3, time
from Crypto.Hash import SHA3_256

with open(os.path.join(LOCAL_DIR, "encrypted_chunks", "chunk_1.enc"), "rb") as f:
    ct = f.read()

N = 5
sha3_times, b3_times = [], []

for _ in range(N):
    t0 = time.perf_counter()
    SHA3_256.new(ct).digest()
    sha3_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    blake3.blake3(ct).digest()
    b3_times.append(time.perf_counter() - t0)

sha3_mean = sum(sha3_times)/N * 1000
b3_mean   = sum(b3_times)/N * 1000
speedup   = sha3_mean / b3_mean

print(f"Chunk size: {len(ct)/1e6:.1f} MB  (N={N} runs each)\n")
print(f"SHA3-256:  {sha3_mean:.1f} ms")
print(f"BLAKE3:    {b3_mean:.1f} ms")
print(f"Speedup:   {speedup:.1f}x")
print(f"Time saved per chunk: {sha3_mean - b3_mean:.1f} ms")
print(f"Projected total saving (12 chunks): {(sha3_mean - b3_mean)*12/1000:.1f}s")

out = "/content/drive/MyDrive/PQC_ModelServing/results/hash_benchmark.csv"
import csv
with open(out, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["algorithm", "mean_ms", "runs", "chunk_mb"])
    w.writerow(["SHA3-256", f"{sha3_mean:.2f}", N, f"{len(ct)/1e6:.1f}"])
    w.writerow(["BLAKE3",   f"{b3_mean:.2f}",   N, f"{len(ct)/1e6:.1f}"])
    w.writerow(["speedup",  f"{speedup:.2f}",    "", ""])
print(f"Saved: {out}")
del ct

# logging

In [ ]:
import threading, collections, time, os, json, gc
import torch, blake3, ctypes, base64, io
import oqs, psutil
from datetime import datetime, timezone
from Crypto.Cipher import AES
from Crypto.Hash import SHA3_256
from Crypto.Util.strxor import strxor
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer

ACCESS_LOG_PATH = "/content/drive/MyDrive/PQC_ModelServing/results/access_log.jsonl"
os.makedirs(os.path.dirname(ACCESS_LOG_PATH), exist_ok=True)


def log_event(event: dict):
    event["ts"] = datetime.now(timezone.utc).isoformat()
    with open(ACCESS_LOG_PATH, "a") as f:
        f.write(json.dumps(event) + "\n")


class ChunkAccessDetector:
    def __init__(self, num_chunks: int = 12,
                 max_repeats: int = 2,
                 max_rpm: int = 10,
                 enforce_order: bool = True):
        self.num_chunks    = num_chunks
        self.max_repeats   = max_repeats
        self.max_rpm       = max_rpm
        self.enforce_order = enforce_order
        self.history       = collections.defaultdict(list)
        self.blocked       = set()
        self.timestamps    = collections.deque()
        self.incidents     = []
        self._lock         = threading.Lock()

    def _incident(self, req_id, chunk_id, reason, history):
        rec = {"req_id": req_id, "chunk_id": chunk_id,
               "reason": reason, "history": list(history),
               "ts": datetime.now(timezone.utc).isoformat()}
        self.incidents.append(rec)
        log_event({"event": "INCIDENT", **rec})
        print(f"\n[ANOMALY] {req_id} chunk={chunk_id} — {reason}")

    def check(self, req_id: str, chunk_id: int):
        with self._lock:

            # Check 1: BLOCKED_REQUEST
            if req_id in self.blocked:
                self._incident(req_id, chunk_id,
                    "BLOCKED_REQUEST", self.history[req_id])
                raise RuntimeError(f"[BLOCKED] {req_id} is suspended")

            hist = self.history[req_id]

            # Check 2: OUT_OF_ORDER
            if self.enforce_order:
                expected = len(hist)
                if chunk_id != expected:
                    self.blocked.add(req_id)
                    self._incident(req_id, chunk_id,
                        f"OUT_OF_ORDER: expected chunk {expected} got {chunk_id}",
                        hist)
                    raise RuntimeError(f"[BLOCKED] out-of-order access")

            # Check 3: CHUNK_REPEAT
            if hist.count(chunk_id) >= self.max_repeats:
                self.blocked.add(req_id)
                self._incident(req_id, chunk_id,
                    f"CHUNK_REPEAT: chunk {chunk_id} accessed "
                    f"{hist.count(chunk_id)+1}x (max={self.max_repeats})",
                    hist)
                raise RuntimeError(
                    f"[BLOCKED] chunk {chunk_id} repeated too many times")

            # Check 4: RATE_LIMIT
            now = time.monotonic()
            self.timestamps.append(now)
            while self.timestamps and now - self.timestamps[0] > 60:
                self.timestamps.popleft()
            if len(self.timestamps) > self.max_rpm:
                self._incident(req_id, chunk_id,
                    f"RATE_LIMIT: {len(self.timestamps)} req/min "
                    f"(max={self.max_rpm})",
                    hist)

            self.history[req_id].append(chunk_id)
            log_event({"event": "ACCESS", "req_id": req_id,
                       "chunk_id": chunk_id,
                       "ram_gb": psutil.Process(os.getpid()).memory_info().rss/1e9})

    def summarize(self):
        print(f"\n{'='*50}")
        print(f"Requests tracked: {len(self.history)}")
        print(f"Blocked:          {len(self.blocked)}")
        print(f"Incidents:        {len(self.incidents)}")
        for inc in self.incidents:
            print(f"  {inc['ts']}  {inc['req_id']:20s}  {inc['reason']}")


def decrypt_chunk_monitored(chunk_id: int, req_id: str,
                             store_dir: str,
                             kem_secret_key: bytes,
                             sig_public_key: bytes,
                             detector: ChunkAccessDetector) -> dict:
    detector.check(req_id, chunk_id)

    t0 = time.perf_counter()
    with open(os.path.join(store_dir, "encrypted_chunks",
                           f"chunk_{chunk_id}.enc"), "rb") as f:
        ciphertext = f.read()
    with open(os.path.join(store_dir, "key_manifest.json")) as f:
        km = json.load(f)
    with open(os.path.join(store_dir, "signatures.json")) as f:
        sigs = json.load(f)

    entry = km[str(chunk_id)]
    digest = blake3.blake3(ciphertext).digest()
    sig    = base64.b64decode(sigs[str(chunk_id)])

    if not oqs.Signature(SIG_NAME).verify(digest, sig, sig_public_key):
        log_event({"event": "SIG_FAIL", "req_id": req_id, "chunk_id": chunk_id})
        raise ValueError(f"Chunk {chunk_id}: integrity check failed")

    kem = oqs.KeyEncapsulation(KEM_NAME, secret_key=kem_secret_key)
    shared  = kem.decap_secret(base64.b64decode(entry["kem_ciphertext"]))
    aes_key = strxor(base64.b64decode(entry["encrypted_aes_key"]),
                     SHA3_256.new(shared).digest())
    nonce   = base64.b64decode(entry["nonce"])
    tag     = base64.b64decode(entry["tag"])

    plaintext = AES.new(aes_key, AES.MODE_GCM,
                        nonce=nonce).decrypt_and_verify(ciphertext, tag)
    del ciphertext

    decrypt_ms = (time.perf_counter() - t0) * 1000
    log_event({"event": "DECRYPTED", "req_id": req_id,
               "chunk_id": chunk_id, "decrypt_ms": round(decrypt_ms, 2),
               "bytes": len(plaintext)})

    return torch.load(io.BytesIO(plaintext), weights_only=False)


def secure_inference(prompt: str, req_id: str,
                     store_dir: str,
                     kem_secret_key: bytes, sig_public_key: bytes,
                     detector: ChunkAccessDetector,
                     model, empty_state: dict, tokenizer,
                     max_tokens: int = 20):
    for chunk_id in range(manifest["num_chunks"]):
        try:
            tensors = decrypt_chunk_monitored(chunk_id, req_id,
                          store_dir, kem_secret_key, sig_public_key, detector)
        except RuntimeError as e:
            log_event({"event": "BLOCKED", "req_id": req_id,
                       "chunk_id": chunk_id, "reason": str(e)})
            return None

        with torch.no_grad():
            for k, v in tensors.items():
                empty_state[k].copy_(v)
        for v in tensors.values():
            secure_wipe(v)
        del tensors

    gc.collect()
    model.load_state_dict(empty_state)
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens,
                              do_sample=False, temperature=None, top_p=None)
    return tokenizer.decode(out[0], skip_special_tokens=True)


print("Logging + detection system loaded.")
print(f"Log: {ACCESS_LOG_PATH}")

In [ ]:
import json, os

config = AutoConfig.from_pretrained(manifest["model_name"])
inf_model  = AutoModelForCausalLM.from_config(config, dtype=torch.float16)
inf_model.eval()
inf_state  = inf_model.state_dict()
inf_tok    = AutoTokenizer.from_pretrained(manifest["model_name"])

detector = ChunkAccessDetector(
    num_chunks=manifest["num_chunks"],
    max_repeats=2,
    max_rpm=10,
    enforce_order=True
)

#Test 1: LEGITIMATE
print(f"{'='*55}")
print("TEST 1: LEGITIMATE (full sequential load, all 12 chunks)")
print(f"{'='*55}")
try:
    result = secure_inference(
        "Post-quantum cryptography secures",
        "req_legit_001", LOCAL_DIR,
        KEM_SECRET, SIG_PUBLIC_KEY,
        detector, inf_model, inf_state, inf_tok)
    print(f"Output: {result}")
    print("Result: PASSED — no anomalies detected")
except Exception as e:
    print(f"Unexpected error: {e}")

#Test 2: OUT_OF_ORDER
print(f"\n{'='*55}")
print("TEST 2: OUT_OF_ORDER (first access jumps directly to chunk 6)")
print(f"{'='*55}")
try:
    detector.check("req_atk_ooo", 6)
    print("ERROR: Should have been blocked — fix not applied")
except RuntimeError as e:
    print(f"Correctly blocked: {e}")

#Test 3: CHUNK_REPEAT
print(f"\n{'='*55}")
print("TEST 3: CHUNK_REPEAT (replay chunk 0 beyond max_repeats=2)")
print(f"{'='*55}")
repeat_detector = ChunkAccessDetector(
    num_chunks=12, max_repeats=2, max_rpm=100, enforce_order=False
)
for attempt in range(4):
    try:
        repeat_detector.check("req_atk_rep", 0)
        print(f"  Attempt {attempt+1} (chunk 0): allowed")
    except RuntimeError as e:
        print(f"  Attempt {attempt+1} (chunk 0): correctly blocked — {e}")
        break

#Test 4: BLOCKED_REQUEST
print(f"\n{'='*55}")
print("TEST 4: BLOCKED_REQUEST (req_atk_ooo already blocked, retries)")
print(f"{'='*55}")
try:
    detector.check("req_atk_ooo", 0)
    print("ERROR: Should have been blocked — fix not applied")
except RuntimeError as e:
    print(f"Correctly blocked: {e}")

#Test 5: RATE_LIMIT
print(f"\n{'='*55}")
print("TEST 5: RATE_LIMIT (burst of 8 rapid requests, max_rpm=5)")
print(f"{'='*55}")
rate_detector = ChunkAccessDetector(
    num_chunks=12, max_repeats=100, max_rpm=5, enforce_order=False
)
for i in range(8):
    rate_detector.check(f"req_burst_{i}", 0)
    if rate_detector.incidents:
        print(f"  Request {i+1}: RATE_LIMIT incident fired")
        break
    else:
        print(f"  Request {i+1}: allowed")

detector.summarize()

incident_path = "/content/drive/MyDrive/PQC_ModelServing/results/incident_report.json"
with open(incident_path, "w") as f:
    json.dump(detector.incidents, f, indent=2)
print(f"\nIncidents saved: {incident_path}")

print("\nAccess log tail (last 8 events):")
with open(ACCESS_LOG_PATH) as f:
    lines = f.readlines()
for line in lines[-8:]:
    ev = json.loads(line)
    print(f"  {ev.get('ts','')}  {ev.get('event',''):12s}  "
          f"req={ev.get('req_id','?'):20s}  chunk={ev.get('chunk_id','?')}")

# Person C - Benchmarking

In [ ]:
import csv
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

results_dir = "/content/drive/MyDrive/PQC_ModelServing/results"
figures_dir = os.path.join(results_dir, "figures")
os.makedirs(figures_dir, exist_ok=True)

print("Person C setup complete.")
print(f"Results dir: {results_dir}")
print(f"Figures dir: {figures_dir}")

In [ ]:
runs        = [1, 2, 3, 4, 5]
run_times   = [76.90, 78.35, 77.44, 77.02, 77.20]
mean_t      = 77.38
std_t       = 0.58
baseline    = 60.98

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(runs, run_times, color="#4C72B0", width=0.5,
              zorder=3, label="PQC Chunked Decrypt")

ax.axhline(baseline, color="#DD4444", linewidth=2,
           linestyle="--", zorder=4, label=f"Baseline (plaintext load): {baseline:.2f}s")

ax.axhline(mean_t, color="#2CA02C", linewidth=2,
           linestyle="-.", zorder=4, label=f"Mean: {mean_t:.2f}s ± {std_t:.2f}s (CV=0.7%)")

for bar, t in zip(bars, run_times):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{t:.2f}s", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xlabel("Run", fontsize=12)
ax.set_ylabel("Total Decryption Time (s)", fontsize=12)
ax.set_title("Latency Benchmark — PQC Chunked Model Serving\nMistral-7B, 12 Chunks, Local Disk, Cache-Warm",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 100)
ax.set_xticks(runs)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3, zorder=0)

ax.annotate(f"+{100*(mean_t/baseline-1):.1f}% overhead",
            xy=(5, mean_t), xytext=(4.2, 90),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=10, color="black")

plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "fig1_latency_benchmark.pdf"), dpi=300)
plt.savefig(os.path.join(figures_dir, "fig1_latency_benchmark.png"), dpi=300)
plt.show()
print("Saved: fig1_latency_benchmark")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

categories  = ["Baseline\n(Plaintext Load)", "PQC Chunked\n(Ours)"]
exposures   = [14483.5, 1833.0]
colors      = ["#DD4444", "#4C72B0"]

bars = axes[0].bar(categories, exposures, color=colors, width=0.4, zorder=3)
axes[0].set_ylabel("Peak Plaintext Exposure (MB)", fontsize=11)
axes[0].set_title("Peak Plaintext Exposure\nat Any Instant", fontsize=11,
                   fontweight="bold")
axes[0].set_ylim(0, 17000)
axes[0].grid(axis="y", alpha=0.3, zorder=0)

for bar, val in zip(bars, exposures):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 150,
                 f"{val:,.0f} MB", ha="center", fontsize=10, fontweight="bold")

axes[0].text(1, 1833 + 900, "↓ 87.3%\nreduction",
             ha="center", fontsize=11, color="#2CA02C", fontweight="bold")

ps_values = [909701, 94731]

bars2 = axes[1].bar(categories, ps_values, color=colors, width=0.4, zorder=3)
axes[1].set_ylabel("Plaintext-Seconds (MB·s)", fontsize=11)
axes[1].set_title("Plaintext-Seconds\n(Temporal Exposure Metric)", fontsize=11,
                   fontweight="bold")
axes[1].set_ylim(0, 1100000)
axes[1].grid(axis="y", alpha=0.3, zorder=0)

for bar, val in zip(bars2, ps_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10000,
                 f"{val:,.0f}", ha="center", fontsize=10, fontweight="bold")

axes[1].text(1, 94731 + 60000, "↓ 89.6%\nreduction",
             ha="center", fontsize=11, color="#2CA02C", fontweight="bold")

fig.suptitle("Attack Surface Comparison — Mistral-7B (14.48 GB)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "fig2_attack_surface.pdf"), dpi=300)
plt.savefig(os.path.join(figures_dir, "fig2_attack_surface.png"), dpi=300)
plt.show()
print("Saved: fig2_attack_surface")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

stages = ["File Read", "BLAKE3\nHash", "ML-DSA-65\nVerify",
          "ML-KEM-768\nDecap", "AES-256\nGCM", "Torch\nDeserialize"]
times  = [4909.6, 309.7, 0.3, 0.3, 4843.2, 647.7]
colors = ["#4C72B0", "#55A868", "#C44E52", "#C44E52", "#8172B2", "#CCB974"]

bars = ax.bar(stages, times, color=colors, zorder=3, width=0.55)

for bar, val in zip(bars, times):
    label = f"{val:.1f}ms"
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 50,
            label, ha="center", fontsize=9, fontweight="bold")

ax.text(0.72, 0.85,
        "PQC Total\n(ML-DSA-65 + ML-KEM-768)\n0.6ms = 0.006% of total",
        transform=ax.transAxes,
        fontsize=9, color="#C44E52", fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="#C44E52", linewidth=1.5))

ax.set_ylabel("Time (ms)", fontsize=12)
ax.set_title("Per-Chunk Decrypt Stage Breakdown\nChunk 1 (1308.7 MB typical)",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 6500)
ax.grid(axis="y", alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig(os.path.join(figures_dir, "fig3_stage_breakdown.pdf"), dpi=300,
            bbox_inches="tight")
plt.savefig(os.path.join(figures_dir, "fig3_stage_breakdown.png"), dpi=300,
            bbox_inches="tight")
plt.show()
print("Saved: fig3_stage_breakdown")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

algos  = ["SHA3-256\n(Original)", "BLAKE3\n(Ours)"]
times  = [5991.3, 295.0]
colors = ["#DD4444", "#2CA02C"]

bars = ax.bar(algos, times, color=colors, width=0.35, zorder=3)

for bar, val in zip(bars, times):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 60,
            f"{val:.1f} ms", ha="center", fontsize=12, fontweight="bold")

ax.set_ylabel("Hash Time (ms)", fontsize=12)
ax.set_title("Integrity Hash Benchmark\n1308.7 MB chunk, N=5 runs",
             fontsize=12, fontweight="bold")
ax.set_ylim(0, 7500)
ax.grid(axis="y", alpha=0.3, zorder=0)

ax.text(0.55, 0.65,
        "20.3× faster\nSaves 68.4s per\nfull model load",
        transform=ax.transAxes,
        fontsize=11, color="#2CA02C", fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="#2CA02C", linewidth=1.5))

plt.savefig(os.path.join(figures_dir, "fig4_hash_benchmark.pdf"), dpi=300,
            bbox_inches="tight")
plt.savefig(os.path.join(figures_dir, "fig4_hash_benchmark.png"), dpi=300,
            bbox_inches="tight")
plt.show()
print("Saved: fig4_hash_benchmark")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

events = [
    (0,  "req_legit_001", "ACCESS",            "#4C72B0", "o", 120),
    (1,  "req_legit_001", "ACCESS",            "#4C72B0", "o", 120),
    (11, "req_legit_001", "DECRYPTED",         "#2CA02C", "o", 180),
    (12, "req_atk_ooo",  "OUT_OF_ORDER",       "#DD4444", "D", 250),
    (13, "req_atk_rep",  "ACCESS",             "#4C72B0", "o", 120),
    (14, "req_atk_rep",  "ACCESS",             "#4C72B0", "o", 120),
    (15, "req_atk_rep",  "CHUNK_REPEAT",       "#DD4444", "D", 250),
    (16, "req_atk_ooo",  "BLOCKED_REQUEST",    "#8B0000", "D", 300),
    (17, "req_burst_5",  "RATE_LIMIT",         "#FF8C00", "D", 250),
]

y_map = {
    "req_legit_001": 3,
    "req_atk_ooo":   2,
    "req_atk_rep":   1,
    "req_burst_5":   0,
}

y_labels = ["req_burst (Rate Limit)", "req_atk_rep (Chunk Repeat)",
            "req_atk_ooo (Out-of-Order)", "req_legit (Legitimate)"]

for x, req, event, color, marker, size in events:
    ax.scatter(x, y_map[req], color=color, s=size, marker=marker,
               zorder=3, edgecolors="white", linewidths=0.5)
    if event not in ("ACCESS", "DECRYPTED"):
        ax.text(x + 0.3, y_map[req] + 0.18, event,
                fontsize=8.5, color=color, fontweight="bold")

ax.set_yticks([0, 1, 2, 3])
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel("Event Sequence", fontsize=11)
ax.set_title("Anomaly Detection — All 4 Attack Patterns Detected",
             fontsize=12, fontweight="bold")
ax.set_ylim(-0.5, 3.8)
ax.set_xlim(-1, 22)
ax.grid(alpha=0.3)

legend_elements = [
    mpatches.Patch(color="#4C72B0", label="ACCESS (allowed)"),
    mpatches.Patch(color="#2CA02C", label="DECRYPTED (complete)"),
    mpatches.Patch(color="#DD4444", label="OUT_OF_ORDER / CHUNK_REPEAT"),
    mpatches.Patch(color="#8B0000", label="BLOCKED_REQUEST"),
    mpatches.Patch(color="#FF8C00", label="RATE_LIMIT"),
]
ax.legend(handles=legend_elements, fontsize=8.5,
          loc="lower left", framealpha=0.9)

plt.savefig(os.path.join(figures_dir, "fig5_anomaly_detection.pdf"), dpi=300,
            bbox_inches="tight")
plt.savefig(os.path.join(figures_dir, "fig5_anomaly_detection.png"), dpi=300,
            bbox_inches="tight")
plt.show()
print("Saved: fig5_anomaly_detection")

In [ ]:
print("\n" + "="*65)
print("TABLE 1 — SYSTEM SUMMARY")
print("="*65)
print(f"{'Metric':<40} {'Value':>20}")
print("-"*65)
rows = [
    ("Model",                           "Mistral-7B-v0.1 (14.48 GB)"),
    ("Chunks",                          "12 balanced chunks"),
    ("KEM Algorithm",                   "ML-KEM-768 (FIPS 203)"),
    ("Signature Algorithm",             "ML-DSA-65 (FIPS 204)"),
    ("Integrity Hash",                  "BLAKE3"),
    ("Encryption",                      "AES-256-GCM"),
    ("Baseline load time",              "62.81s"),
    ("Mean chunked decrypt time",       "77.38s ± 0.58s"),
    ("Coefficient of variation",        "0.7%"),
    ("Latency overhead",                "+26.9%"),
    ("Peak plaintext exposure",         "1833.0 MB (12.7%)"),
    ("Attack surface reduction",        "87.3%"),
    ("Plaintext-seconds (baseline)",    "909,701 MB·s"),
    ("Plaintext-seconds (chunked)",     "94,731 MB·s"),
    ("Plaintext-seconds reduction",     "89.6%"),
    ("PQC overhead per chunk",          "0.6ms (0.006%)"),
    ("BLAKE3 vs SHA3-256 speedup",      "20.3×"),
    ("Correctness (5 prompts)",         "PASSED"),
    ("Anomaly detection modes",         "4 / 4 verified"),
]
for metric, value in rows:
    print(f"{metric:<40} {value:>20}")
print("="*65)

table_path = os.path.join(results_dir, "table1_summary.csv")
with open(table_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["metric", "value"])
    w.writerows(rows)
print(f"\nSaved: {table_path}")

In [ ]:
print("\n" + "="*80)
print("TABLE 2 — COMPARISON WITH RELATED WORK")
print("="*80)
print(f"{'System':<20} {'PQC':>6} {'No TEE':>7} {'Runtime Isolation':>18} "
      f"{'Overhead':>10} {'Model':>12}")
print("-"*80)
related = [
    ("CryptoTensors",  "No",  "Yes", "No",  "+61-62%",  "GPT-2/BERT"),
    ("PipeLLM",        "No",  "No",  "Yes", "~15%",     "LLaMA"),
    ("FlexServe",      "No",  "No",  "Yes", "varies",   "Mobile LLMs"),
    ("Chutes E2EE",    "Yes", "No",  "No",  "minimal",  "Prompt only"),
    ("Ours",           "Yes", "Yes", "Yes", "+26.9%",   "Mistral-7B"),
]
for row in related:
    print(f"{row[0]:<20} {row[1]:>6} {row[2]:>7} {row[3]:>18} "
          f"{row[4]:>10} {row[5]:>12}")
print("="*80)

table2_path = os.path.join(results_dir, "table2_related_work.csv")
with open(table2_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["system", "pqc", "no_tee", "runtime_isolation",
                "overhead", "model"])
    w.writerows(related)
print(f"Saved: {table2_path}")